#SQL Analysis - Retail Orders Dataset

This notebook uses SQL queries to analyze the cleaned retail orders data stored in a SQLite database.

Main analysis areas:
1. Total orders and revenue
2. Order status analysis
3. Product revenue analysis
4. Payment method analysis
5. Referral source analysis
6. Coupon usage analysis
7. Monthly revenue trend
8. Customer order analysis

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
database_path = Path("../database/orders_cleaned.sqlite")

connection = sqlite3.connect(database_path)

print("Database connected successfully")

Database connected successfully


In [3]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    connection
)

tables

,name
0,cleaned_orders


In [5]:
#total orders, total revenue, Average Order Value
query = """
SELECT 
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_revenue,
    ROUND(AVG(TotalPrice), 2) AS average_order_value
FROM cleaned_orders;
"""

total_summary = pd.read_sql_query(query, connection)
total_summary

,total_orders,total_revenue,average_order_value
0,1200,1264761.96,1053.97


In [6]:
#orders by status
query = """
SELECT 
    OrderStatus,
    COUNT(*) AS order_count,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM cleaned_orders
GROUP BY OrderStatus
ORDER BY order_count DESC;
"""

orders_by_status = pd.read_sql_query(query, connection)
orders_by_status

,OrderStatus,order_count,total_revenue
0,Cancelled,250,276396.21
1,Returned,247,243277.70
2,Pending,237,256328.15
3,Shipped,235,246159.58
4,Delivered,231,242600.32


In [7]:
#revenue by product
query = """
SELECT 
    Product,
    COUNT(*) AS total_orders,
    SUM(Quantity) AS total_quantity_sold,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM cleaned_orders
GROUP BY Product
ORDER BY total_revenue DESC;
"""

revenue_by_product = pd.read_sql_query(query, connection)
revenue_by_product

,Product,total_orders,total_quantity_sold,total_revenue
0,Chair,178,562,195620.11
1,Printer,181,542,195612.61
2,Laptop,173,535,192126.56
3,Tablet,179,497,186568.95
4,Monitor,163,480,175651.41
5,Desk,170,508,167459.93
6,Phone,156,411,151722.39


In [8]:
#payment method analysis
query = """
SELECT 
    PaymentMethod,
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_revenue,
    ROUND(AVG(TotalPrice), 2) AS average_order_value
FROM cleaned_orders
GROUP BY PaymentMethod
ORDER BY total_orders DESC;
"""

payment_analysis = pd.read_sql_query(query, connection)
payment_analysis

,PaymentMethod,total_orders,total_revenue,average_order_value
0,Online,258,262442.94,1017.22
1,Cash,246,259786.29,1056.04
2,Credit Card,234,263847.63,1127.55
3,Debit Card,232,232361.18,1001.56
4,Gift Card,230,246323.92,1070.97


In [9]:
#referral source analysis
query = """
SELECT 
    ReferralSource,
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM cleaned_orders
GROUP BY ReferralSource
ORDER BY total_orders DESC;
"""

referral_analysis = pd.read_sql_query(query, connection)
referral_analysis

,ReferralSource,total_orders,total_revenue
0,Instagram,259,275285.45
1,Email,250,261808.55
2,Google,241,250441.48
3,Facebook,228,250410.90
4,Referral,222,226815.58


In [10]:
#coupon usage analysis
query = """
SELECT 
    CASE 
        WHEN CouponCode = 'NO_COUPON' THEN 'No Coupon'
        ELSE 'Used Coupon'
    END AS coupon_status,
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM cleaned_orders
GROUP BY coupon_status
ORDER BY total_orders DESC;
"""

coupon_usage = pd.read_sql_query(query, connection)
coupon_usage

,coupon_status,total_orders,total_revenue
0,Used Coupon,891,942360.55
1,No Coupon,309,322401.41


In [11]:
#monthly revenue trend
query = """
SELECT 
    strftime('%Y-%m', Date) AS month,
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM cleaned_orders
GROUP BY month
ORDER BY month;
"""

monthly_revenue = pd.read_sql_query(query, connection)
monthly_revenue

,month,total_orders,total_revenue
0,2023-01,47,56685.75
1,2023-02,37,40117.66
2,2023-03,43,48609.37
3,2023-04,31,27751.71
4,2023-05,49,63836.84
5,2023-06,45,49500.19
6,2023-07,44,42820.66
7,2023-08,51,54352.14
8,2023-09,29,29526.67
9,2023-10,47,52607.85


In [12]:
#top customers by revenue
query = """
SELECT 
    CustomerID,
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_spent
FROM cleaned_orders
GROUP BY CustomerID
ORDER BY total_spent DESC
LIMIT 10;
"""

top_customers = pd.read_sql_query(query, connection)
top_customers

,CustomerID,total_orders,total_spent
0,C38840,2,5723.23
1,C57276,1,3456.40
2,C67260,1,3390.80
3,C13877,1,3384.90
4,C18404,1,3370.20
5,C16775,1,3353.75
6,C65986,1,3352.40
7,C47778,1,3334.00
8,C59183,1,3322.55
9,C25276,1,3313.90


In [13]:
connection.close() #close database connection

print("Database connection closed")

Database connection closed
